# Physical systems and modelsExecutable companion to chapter 4. Four schematic models, each simple enoughto solve exactly and each containing a genuine many-body mechanism:| Model | Interaction | Symmetry used ||---|---|---|| Lipkin | pair transfer + spin exchange | $SU(2)$ quasispin || Pairing | pair transfer | $S$, $S_z$, seniority || Fermi-Hubbard | on-site $U$ | $N_\uparrow$, $N_\downarrow$ || Calogero | inverse square | integrability |Each is the general second-quantised Hamiltonian with a drastically restrictedset of two-body matrix elements. In every case a symmetry reduces thedimension of the problem to something we can diagonalise — which is why theseare the benchmarks against which approximate methods are measured.

In [ ]:
from itertools import combinationsimport numpy as npimport matplotlib.pyplot as pltnp.set_printoptions(precision=6, suppress=True)

## 1. The Lipkin modelTwo levels of degeneracy $\Omega$, quantum numbers $\sigma=\pm1$ and$p=1,\dots,\Omega$. Because the interaction is completely degenerate in $p$,the model has an $SU(2)$ quasispin symmetry and the Hamiltonian collapses to$$H = \varepsilon J_z + \tfrac12 V (J_+^2 + J_-^2)  + \tfrac12 W (-N + J_+J_- + J_-J_+),$$which is block diagonal in $J$. For four particles the largest block is$J=2$: a 70-dimensional problem becomes a $5\times5$ matrix.

In [ ]:
class LipkinModel:    """The Lipkin model in one J block of the quasispin basis."""    def __init__(self, J=2.0, eps=2.0, V=-1/3, W=-1/4, N=None):        self.J, self.eps, self.V, self.W = J, eps, V, W        self.N = 2*J if N is None else N        self.jz = np.arange(-J, J + 1)        self.dim = len(self.jz)    @staticmethod    def _jplus(J, m):        """<J,m+1| J_+ |J,m> = sqrt(J(J+1) - m(m+1))."""        v = J*(J + 1) - m*(m + 1)        return np.sqrt(v) if v > 0 else 0.0    def matrix(self):        J, dim = self.J, self.dim        H = np.zeros((dim, dim))        for a, m in enumerate(self.jz):            H[a, a] += self.eps*m                                   # eps J_z            # J_+J_- + J_-J_+ = 2(J(J+1) - J_z^2), so this term is diagonal            H[a, a] += 0.5*self.W*(-self.N + 2*(J*(J+1) - m*m))            if a + 2 < dim:                       # (V/2)(J_+^2 + J_-^2)                e = 0.5*self.V*self._jplus(J, m)*self._jplus(J, m+1)                H[a+2, a] += e                H[a, a+2] += e        return Hfor eps, V, W in ((2.0, -1/3, -1/4), (2.0, -4/3, -1.0)):    m = LipkinModel(J=2.0, eps=eps, V=V, W=W, N=4)    vals, vecs = np.linalg.eigh(m.matrix())    print(f"eps = {eps}, V = {V:+.4f}, W = {W:+.4f}")    print("  eigenvalues:", np.round(vals, 5))    psi = vecs[:, 0]    parts = [f"{abs(psi[k]):.5f}|2,{int(mm):+d}>"             for k, mm in enumerate(m.jz) if abs(psi[k]) > 1e-3]    print(f"  E_0 = {vals[0]:.5f}   " + " + ".join(parts))    print()

### The single-configuration picture breaking downAt weak coupling the unperturbed configuration $|2,-2\rangle$ carries nearlyall the probability and one Slater determinant suffices. As $V$ grows, weightshifts to $|2,0\rangle$ and no single determinant will do — the same story theoccupation numbers of chapter 1 told, in a model small enough to see all of.

In [ ]:
Vs = np.linspace(0.0, -3.0, 61)E0, weight = [], []for V in Vs:    m = LipkinModel(J=2.0, eps=2.0, V=V, W=-1/4, N=4)    vals, vecs = np.linalg.eigh(m.matrix())    E0.append(vals[0])    weight.append(vecs[0, 0]**2)          # |<2,-2|psi_0>|^2fig, ax = plt.subplots(1, 2, figsize=(10, 4))ax[0].plot(-Vs, E0)ax[0].set_xlabel("$-V$")ax[0].set_ylabel("$E_0$")ax[0].set_title("Lipkin ground-state energy")ax[0].grid(alpha=0.3)ax[1].plot(-Vs, weight)ax[1].set_xlabel("$-V$")ax[1].set_ylabel(r"$|\langle 2,-2|\psi_0\rangle|^2$")ax[1].set_title("Weight of the unperturbed configuration")ax[1].grid(alpha=0.3)plt.tight_layout()plt.show()

## 2. The pairing model$L$ doubly degenerate, equally spaced levels; the interaction destroys a pairin level $q$ and creates one in level $p$, and does nothing else:$$H = \xi\sum_{p\sigma}(p-1)a^\dagger_{p\sigma}a_{p\sigma}  - \tfrac12 g \sum_{pq} P^+_p P^-_q ,\qquad P^+_p = a^\dagger_{p+}a^\dagger_{p-}.$$Since the interaction never breaks a pair, the seniority-zero space — onebasis state per choice of which levels carry a pair — is closed. Its dimensionis $\binom{L}{n}$ rather than $\binom{2L}{2n}$: 6 instead of 28 for $L=4$,$n=2$; 924 instead of 2 704 156 for $L=12$, $n=6$.

In [ ]:
class PairingModel:    """The pairing model in the seniority-zero space."""    def __init__(self, levels=4, pairs=2, g=1.0, xi=1.0):        self.levels, self.pairs, self.g, self.xi = levels, pairs, g, xi        self.basis = list(combinations(range(1, levels + 1), pairs))        self.index = {c: i for i, c in enumerate(self.basis)}        self.dim = len(self.basis)    def matrix(self):        H = np.zeros((self.dim, self.dim))        for config, i in self.index.items():            occ = set(config)            H[i, i] += 2*self.xi*sum(p - 1 for p in config)   # unperturbed            H[i, i] -= 0.5*self.g*self.pairs                  # p = q terms            for q in config:                                  # move one pair                for p in range(1, self.levels + 1):                    if p in occ:                        continue                    H[self.index[tuple(sorted(occ - {q} | {p}))], i] -= 0.5*self.g        return H    def ground_state_energy(self):        return float(np.linalg.eigvalsh(self.matrix())[0])    def reference_energy(self):        i = self.index[tuple(range(1, self.pairs + 1))]        return float(self.matrix()[i, i])print(f"{'g':>7s} {'E_ref':>12s} {'E_0 (exact)':>14s} {'E_corr':>14s}")for g in (-1.0, -0.5, 0.0, 0.5, 1.0):    m = PairingModel(levels=4, pairs=2, g=g)    print(f"{g:7.2f} {m.reference_energy():12.6f} "          f"{m.ground_state_energy():14.8f} "          f"{m.ground_state_energy()-m.reference_energy():14.8f}")

In [ ]:
gs = np.linspace(-1.0, 1.0, 81)for L, n in ((4, 2), (6, 3), (8, 4)):    corr = [PairingModel(L, n, g).ground_state_energy()            - PairingModel(L, n, g).reference_energy() for g in gs]    plt.plot(gs, corr, label=f"L = {L}, n = {n}")plt.xlabel("$g$")plt.ylabel("correlation energy")plt.title("Pairing model: correlation energy against coupling")plt.grid(alpha=0.3)plt.legend()plt.tight_layout()plt.show()

## 3. The Fermi-Hubbard model$$H = -t\sum_{\langle ij\rangle\sigma}   (c^\dagger_{i\sigma}c_{j\sigma} + \mathrm{h.c.}) + U\sum_i n_{i\uparrow}n_{i\downarrow}$$The Hamiltonian conserves $N_\uparrow$ and $N_\downarrow$ separately, and thehopping acts on the two species independently — so the kinetic part is$K_\uparrow \otimes I + I \otimes K_\downarrow$, the tensor product ofchapter 1 used in earnest, and the interaction is diagonal.The fermionic sign is the Jordan-Wigner string: $c^\dagger_i c_j$ picks up theparity of the occupied sites strictly between $i$ and $j$.

In [ ]:
class HubbardChain:    """The 1D Hubbard model in a fixed (N_up, N_down) sector."""    def __init__(self, sites=4, n_up=2, n_down=2, t=1.0, U=4.0, pbc=True):        self.n, self.t, self.U = sites, t, U        self.bonds = [(i, (i+1) % sites) for i in range(sites)]        if not pbc:            self.bonds = self.bonds[:-1]        self.up = self._states(n_up)        self.dn = self._states(n_down)        self.dim = len(self.up)*len(self.dn)    def _states(self, k):        out = []        for occ in combinations(range(self.n), k):            bits = 0            for s in occ:                bits |= 1 << s            out.append(bits)        return sorted(out)    def _hopping(self, states):        idx = {s: i for i, s in enumerate(states)}        K = np.zeros((len(states), len(states)))        for s in states:            for (i, j) in self.bonds:                for (c, d) in ((i, j), (j, i)):          # c^+_c c_d                    if not (s >> d) & 1 or (s >> c) & 1:                        continue                    lo, hi = min(c, d), max(c, d)        # Jordan-Wigner sign                    mask = ((1 << hi) - 1) ^ ((1 << (lo+1)) - 1)                    sign = (-1)**bin(s & mask).count("1")                    K[idx[(s ^ (1 << d)) | (1 << c)], idx[s]] += -self.t*sign        return K    def _double(self):        return np.array([[bin(u & d).count("1") for d in self.dn]                         for u in self.up]).ravel().astype(float)    def matrix(self):        Ku, Kd = self._hopping(self.up), self._hopping(self.dn)        H = np.kron(Ku, np.eye(len(self.dn))) + np.kron(np.eye(len(self.up)), Kd)        return H + self.U*np.diag(self._double())    def ground(self):        vals, vecs = np.linalg.eigh(self.matrix())        psi = vecs[:, 0]        return float(vals[0]), float(psi @ (self._double()*psi))print(f"{'U/t':>6s} {'dim':>6s} {'E_0/t':>14s} {'double occ.':>13s}")for U in (0.0, 2.0, 4.0, 8.0, 16.0):    m = HubbardChain(sites=4, n_up=2, n_down=2, U=U)    e0, docc = m.ground()    print(f"{U:6.1f} {m.dim:6d} {e0:14.8f} {docc:13.6f}")print("\nAt U = 0 the exact free-fermion result is -4t: the single-particle")print("energies are -2t cos k with k = 0, +/- pi/2, pi.")

### Strong coupling: the Heisenberg limitFor $U \gg t$ the doubly occupied states are eliminated perturbatively andwhat remains is$$H_{\rm eff} = J\sum_{\langle ij\rangle}  \left(\mathbf{S}_i\cdot\mathbf{S}_j - \tfrac14 n_i n_j\right),\qquad J = \frac{4t^2}{U}.$$For the four-site ring the Heisenberg part gives $-2J$ and the constant termon four bonds gives $-J$, so $E_0/J \to -3$. This is a real prediction, andthe numbers confirm it.

In [ ]:
print(f"{'U/t':>7s} {'E_0/t':>16s} {'J = 4t^2/U':>13s} {'E_0/J':>10s}")Us = [8.0, 16.0, 32.0, 64.0, 128.0]ratios = []for U in Us:    e0, _ = HubbardChain(sites=4, n_up=2, n_down=2, U=U).ground()    J = 4.0/U    ratios.append(e0/J)    print(f"{U:7.1f} {e0:16.10f} {J:13.8f} {e0/J:10.5f}")plt.figure(figsize=(6.5, 4))plt.semilogx(Us, ratios, "o-")plt.axhline(-3.0, color="k", ls="--", lw=1, label="$-3$ (Heisenberg limit)")plt.xlabel("$U/t$")plt.ylabel("$E_0 / J$")plt.title("The Hubbard ring becomes the Heisenberg antiferromagnet")plt.grid(alpha=0.3, which="both")plt.legend()plt.tight_layout()plt.show()

## 4. The Calogero model$$H = -\tfrac12\sum_i \partial_i^2 + \tfrac12\omega^2\sum_i x_i^2  + \sum_{i<j}\frac{\lambda(\lambda-1)}{(x_i-x_j)^2}$$Unlike the first three, this one is solvable for *any* $N$. The ground stateis a pure Jastrow function,$$\Psi_0 = \prod_{i<j}|x_i-x_j|^\lambda   \exp\!\left(-\tfrac{\omega}{2}\sum_i x_i^2\right),\qquadE_0 = \omega\left[\frac{N}{2} + \frac{\lambda N(N-1)}{2}\right].$$At $\lambda = 1$ the coupling vanishes and the Jastrow factor becomes theVandermonde determinant — the Slater determinant of free fermions.

In [ ]:
def calogero_energy(N, lam, omega=1.0):    return omega*(0.5*N + 0.5*lam*N*(N - 1))print("At lambda = 1 the energy must equal the sum of the N lowest")print("oscillator levels, since the ground state is the free-fermion")print("Slater determinant:\n")print(f"{'N':>4s} {'E_0(lambda=1)':>16s} {'sum (n+1/2)':>14s}")for N in (2, 3, 5, 10):    print(f"{N:4d} {calogero_energy(N, 1.0):16.4f} "          f"{sum(n + 0.5 for n in range(N)):14.4f}")

### A numerical check for two particlesWith $x = x_1 - x_2$ the relative motion separates,$$h_{\rm rel} = -\frac{d^2}{dx^2} + \frac{\omega^2}{4}x^2  + \frac{\lambda(\lambda-1)}{x^2},$$whose exact ground-state energy is $\omega(\lambda + \tfrac12)$. We solve it ona grid with the machinery of chapter 1. The inverse-square barrier keeps thewave function away from the origin, so the boundary condition is automatic.

In [ ]:
def relative_energy(lam, omega=1.0, n_grid=4000, rmax=12.0):    x, h = np.linspace(0.0, rmax, n_grid + 2, retstep=True)    x = x[1:-1]                                     # interior points    diag = 2/h**2 + 0.25*omega**2*x**2 + lam*(lam - 1)/x**2    off = np.full(len(x) - 1, -1/h**2)    H = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)    return float(np.linalg.eigvalsh(H)[0])print(f"{'lambda':>8s} {'numerical':>14s} {'exact':>14s} {'error':>12s}")for lam in (1.0, 1.5, 2.0, 3.0):    num, exact = relative_energy(lam), lam + 0.5    print(f"{lam:8.1f} {num:14.8f} {exact:14.8f} {abs(num-exact):12.2e}")

## What the four have in commonIn every case the route is the same: write the Hamiltonian in secondquantisation, find the operators that commute with it, use them to blockdiagonalise, and solve the blocks. The first step is chapter 3, the second isa commutator calculation that Wick's theorem makes routine, the third is thestatement that a symmetry gives a basis in which the matrix is block diagonal,and the fourth is the eigenvalue problem of chapter 1.What separates these models from a realistic system is only the last step ofthe reduction. In a real nucleus or molecule the symmetries reduce thedimension by orders of magnitude but not to five; the matrix is sparse butenormous; and one must fall back on the approximate methods that occupy therest of the book. The value of these four is that they let us watch thoseapproximations succeed and fail against an answer we already know.